# 02 — Tool layer
Call the same ToolRegistry that is used by both deterministic and OpenAI orchestration.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == '02_notebooks' else Path.cwd().resolve()
SRC = ROOT / '03_src'
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))

import os, json
os.environ['TRAVEL_DATA_MODE']='local'
from travel_agent.tools import ToolRegistry
registry=ToolRegistry()
registry.names

['get_location_info',
 'get_weather',
 'convert_currency',
 'search_hotels',
 'search_attractions',
 'get_transport_options',
 'calculate']

In [2]:
registry.schemas()[0]

{'type': 'function',
 'name': 'get_location_info',
 'description': 'Look up destination metadata from the local city catalogue: country, currency, coordinates, timezone, language and indicative local daily costs.',
 'parameters': {'additionalProperties': False,
  'properties': {'city': {'minLength': 2, 'type': 'string'}},
  'required': ['city'],
  'type': 'object'},
 'strict': True}

In [3]:
examples = {
'get_location_info': {'city':'Vienna'},
'get_weather': {'city':'Vienna','days':3,'unit':'celsius'},
'search_hotels': {'city':'Vienna','nights':3,'max_price_per_night_eur':150,'min_rating':4.0,'top_k':5},
'search_attractions': {'city':'Vienna','categories':['museum','landmark'],'max_ticket_eur':20,'top_k':6},
'get_transport_options': {'city':'Vienna','days':3},
'convert_currency': {'amount':500,'from_currency':'EUR','to_currency':'HUF'},
}
outputs={}
for name,args in examples.items(): outputs[name]=registry.execute(name,args)[0]
outputs

{'get_location_info': {'city': 'Vienna',
  'country': 'Austria',
  'country_code': 'AT',
  'currency': 'EUR',
  'language': 'German',
  'timezone': 'Europe/Vienna',
  'latitude': 48.2082,
  'longitude': 16.3738,
  'daily_food_budget_eur': 42.0,
  'daily_transport_eur': 9.0,
  'tourism_score': 4.7,
  'source': '01_data/raw/cities.csv'},
 'get_weather': {'city': 'Vienna',
  'forecast': [{'date': '2026-09-12',
    'temp_min_c': 13.8,
    'temp_max_c': 22.3,
    'precipitation_mm': 0.0,
    'condition': 'clear',
    'wind_kph': 8.0},
   {'date': '2026-09-13',
    'temp_min_c': 16.3,
    'temp_max_c': 24.8,
    'precipitation_mm': 5.4,
    'condition': 'partly_cloudy',
    'wind_kph': 15.0},
   {'date': '2026-09-14',
    'temp_min_c': 16.9,
    'temp_max_c': 25.4,
    'precipitation_mm': 1.8,
    'condition': 'cloudy',
    'wind_kph': 22.0}],
  'source': '01_data/raw/weather_fallback.csv',
  'fallback': True},
 'search_hotels': {'city': 'Vienna',
  'nights': 3,
  'matched': 15,
  'results':

The tool layer is independent from the LLM. The model selects a function and arguments; Python owns validation, data access and execution.